# Hollywood by the Numbers: Clustering Analysis
- Alex Arce: aarce
- Lunden Mandigo: lundenm
- Tyrone Pettygrue: tpetty

## Preprocessing Data

In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA


In [ ]:
movies = pd.read_csv('movies.csv')

# Drop unecessary columns
movies = movies.drop(columns=['originalTitle', 'main_genre', 'rating_category'])

# Combine the 'title' column and 'year' column to drop duplicate rows but keep movies with the same title
movies['title'] = movies['title'] + ' (' + movies['year'].astype(str) + ')'
movies = movies.drop_duplicates(subset=['title'])

# Make the title column the index

# Split the 'genres' and 'productionCountries' columns from strings into lists
movies['genres'] = movies['genres'].str.split(', ')
movies['productionCountries'] = movies['productionCountries'].str.split(', ')

# Replace NaN values in the new 'genres' and 'productionCountries' columns with empty lists
movies[['genres', 'productionCountries']] = movies[['genres', 'productionCountries']].fillna(value={})

# Drop all values that are not list objects from the 'productionCountries' column
movies['productionCountries'] = movies['productionCountries'].apply(lambda x: x if isinstance(x, list) else ([x] if pd.notna(x) else []))



In [30]:
movies.shape


(4657, 15)

In [14]:
movies.to_csv('movies_cleaned.csv')

In [4]:
movies.productionCountries.value_counts().sort_values(ascending=True).head(10)

productionCountries
[Canada, France, Norway, United Kingdom, United States of America]    1
[Italy, South Africa, United Kingdom, United States of America]       1
[Canada, France, Germany, Switzerland]                                1
[Belgium, France, Germany, Spain, United States of America]           1
[Germany, United States of America, Brazil]                           1
[Italy, Spain]                                                        1
[France, United States of America, Germany]                           1
[United Kingdom, United States of America, Croatia]                   1
[Germany, Canada, United States of America]                           1
[United States of America, Canada, Germany]                           1
Name: count, dtype: int64

In [43]:
# Onehot encode categorical columns with a MultiLabelBinarizer
mlb = MultiLabelBinarizer()
encoded_genres = mlb.fit_transform(movies['genres'])
encoded_genres_df = pd.DataFrame(encoded_genres, columns=mlb.classes_)
encoded_genres_df = encoded_genres_df.fillna(0)
encoded_production_countries = mlb.fit_transform(movies['productionCountries'])
encoded_production_countries_df = pd.DataFrame(encoded_production_countries, columns=mlb.classes_)
encoded_production_countries_df = encoded_production_countries_df.fillna(0)

# add the encoded df columns with the original df on the index
processed_movies = pd.concat([movies, encoded_genres_df], axis=1)
# add the encoded df columns with the original df on the index  
processed_movies = pd.concat([movies, encoded_production_countries_df], axis=1)
# processed_movies = pd.merge(movies, encoded_genres_df, left_index=True, right_index=True)

# concat the encoded columns with the original dataframe
# processed_movies = pd.concat([movies, encoded_genres_df, encoded_production_countries_df], axis=1)

# drop the original columns
processed_movies = processed_movies.drop(columns=['genres'])
processed_movies = processed_movies.drop(columns=['productionCountries'])
# processed_movies = processed_movies.drop(columns=['originalLang'])


processed_movies = movies.set_index('title')



In [44]:
processed_movies.shape

(4657, 14)

In [45]:
Yid = processed_movies.index

In [46]:
categorical_cols = ['originalLang']
numerical_cols = ['runtimeMinutes', 'IMDBavgRating', 'numVotes', 'rank', 'worldwideGross', 'domesticGross', 'domestic%', 'foreignGross', 'foreign%', "year"]


num_pipeline = Pipeline([
    ('impute',SimpleImputer(strategy='median')), 
    ('scale',StandardScaler())
    ])

preprocessing_pipeline = ColumnTransformer([
    ('num', num_pipeline, numerical_cols),
    ('cat', OneHotEncoder(drop='if_binary'), categorical_cols)
    ])

In [47]:
# apply the pipeline to my data
scaled_X = preprocessing_pipeline.fit_transform(processed_movies)

In [48]:
# apply PCA to the scaled data
model = PCA(n_components=7)
X_pca = model.fit_transform(scaled_X)

# create a dataframe from the PCA data
pca_df = pd.DataFrame(X_pca,index=Yid, columns=[f'PC{i}' for i in range(1, 8)])
pca_df.head()
model.explained_variance_ratio_


array([0.37543499, 0.22149613, 0.11904589, 0.0851361 , 0.06657872,
       0.04684637, 0.03722573])

In [49]:
pca_df.head()
pca_df.shape

(4657, 7)

In [50]:
# write the data to a csv
pca_df.to_csv('preprocessed_movie_data.csv')


In [51]:
# apply PCA to the scaled data
model = PCA(n_components=2)
X_pca = model.fit_transform(scaled_X)

# create a dataframe from the PCA data
pca_df = pd.DataFrame(X_pca,index=Yid, columns=[f'PC{i}' for i in range(1, 3)])
pca_df.head()
model.explained_variance_ratio_

pca_df.to_csv('preprocessed_movie_data2.csv')
